# VFD Crystallisation — Identifiability & Sensitivity Analysis

Demonstrates:
1. Fisher information matrix
2. Identifiability diagnostics
3. Sensitivity analysis and ranking
4. Failure cases under low identifiability

In [ ]:
import sys
sys.path.insert(0, '../src')

import numpy as np
import matplotlib.pyplot as plt
from vfd.crystallisation.synthetic_data import ModelParams, ExperimentConfig, generate_vfd_dataset
from vfd.crystallisation.estimation import (
    fisher_information, identifiability_report, sensitivity_analysis, fit_parameters,
)

plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.size'] = 12

## 1. Fisher Information Matrix

In [ ]:
theta = ModelParams(alpha=1.0, beta=0.5, gamma=0.8, eta=0.005)
config = ExperimentConfig(n_modes=3, n_trials=100, n_basins=3, max_steps=500, seed=42)

F = fisher_information(theta, config)
names = ModelParams.param_names()

fig, ax = plt.subplots(figsize=(8, 7))
im = ax.imshow(np.log10(np.abs(F) + 1e-15), cmap='viridis')
ax.set_xticks(range(9))
ax.set_xticklabels(names, rotation=45, ha='right')
ax.set_yticks(range(9))
ax.set_yticklabels(names)
ax.set_title('Fisher Information Matrix (log10 |F_ij|)')
plt.colorbar(im)
plt.tight_layout()
plt.show()

## 2. Identifiability Report

In [ ]:
report = identifiability_report(theta, config)

print(f'Condition number: {report.condition_number:.2e}')
print(f'Identifiable parameters: {report.identifiable_params}')
print(f'Problematic parameters: {report.problematic_params}')
print(f'\nEigenvalues: {np.sort(report.eigenvalues).round(6)}')

# Correlation matrix
fig, ax = plt.subplots(figsize=(8, 7))
im = ax.imshow(report.correlation_matrix, cmap='RdBu_r', vmin=-1, vmax=1)
ax.set_xticks(range(9))
ax.set_xticklabels(names, rotation=45, ha='right')
ax.set_yticks(range(9))
ax.set_yticklabels(names)
ax.set_title('Parameter Correlation Matrix (from Fisher)')
plt.colorbar(im)
plt.tight_layout()
plt.show()

## 3. Sensitivity Analysis

In [ ]:
sens = sensitivity_analysis(theta, config)

print('Parameter sensitivity ranking:')
for name, score in sens.ranking:
    print(f'  {name:10s}: {score:.6f}')

# Sensitivity heatmap
fig, ax = plt.subplots(figsize=(10, 5))
im = ax.imshow(np.abs(sens.normalised_sensitivities), cmap='YlOrRd', aspect='auto')
ax.set_xticks(range(9))
ax.set_xticklabels(names, rotation=45, ha='right')
ax.set_yticks(range(len(sens.observable_names)))
ax.set_yticklabels(sens.observable_names)
ax.set_title('Observable-Parameter Sensitivity Matrix (|normalised|)')
plt.colorbar(im, label='|S_tilde|')
plt.tight_layout()
plt.show()

## 4. Failure Case: Low Identifiability

With very few trials, parameters cannot be uniquely determined.

In [ ]:
config_sparse = ExperimentConfig(n_modes=2, n_trials=5, n_basins=2, max_steps=200, seed=42)
ds_sparse = generate_vfd_dataset(theta, config_sparse)

fit_sparse = fit_parameters(ds_sparse, config_sparse)
print(f'Sparse fit residual: {fit_sparse.residual_norm:.4f}')
print(f'Success: {fit_sparse.success}')

report_sparse = identifiability_report(theta, config_sparse)
print(f'\nCondition number (sparse): {report_sparse.condition_number:.2e}')
print(f'Problematic params (sparse): {report_sparse.problematic_params}')
print('\n=> High condition number and many problematic parameters indicate')
print('   the model is not identifiable with this amount of data.')